In [1]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options

driver = webdriver.Edge(service=Service("C:\\Users\\pmqua\\Downloads\\edgedriver_win64\\msedgedriver.exe"), options=Options())
driver.get('https://tiki.vn/thung-sua-lua-mach-cacao-co-thach-nuvi-100ml-48-hop-x-100ml-phat-trien-chieu-cao-tang-cuong-tri-nho-p278681045.html?itm_campaign=tiki-reco_UNK_DT_UNK_UNK_maybe-you-like_maybe-you-like_pdp-widget-infinity-1-v2-v1_202611081800_MD_batched_PID.278681048&itm_medium=CPC&itm_source=tiki-reco&spid=278681048')
driver.request

SessionNotCreatedException: Message: session not created: This version of Microsoft Edge WebDriver only supports Microsoft Edge version 155
Current browser version is 157.0.4341.42 with binary path C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff678707275]
	(No symbol) [0x7ff678656e34]
	(No symbol) [0x7ff678a96a0f]
	(No symbol) [0x7ff6784224e4]
	(No symbol) [0x7ff67842198b]
	(No symbol) [0x7ff67841e1e8]
	(No symbol) [0x7ff678419820]
	(No symbol) [0x7ff67845bea9]
	(No symbol) [0x7ff67845b806]
	(No symbol) [0x7ff67842590c]
	(No symbol) [0x7ff678424b8d]
	(No symbol) [0x7ff678425733]
	(No symbol) [0x7ff678561a91]
	(No symbol) [0x7ff678571a78]
	(No symbol) [0x7ff6787aff6a]
	(No symbol) [0x7ff6787b65d1]
	(No symbol) [0x7ff6786625dc]
	(No symbol) [0x7ff

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType, ArrayType

In [2]:
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

In [3]:
spark.read.format("kafka").option("kafka.bootstrap.servers", "kafka-1:9092").option("subscribe", "user_review").load().selectExpr("CAST(value AS STRING)").show(5, truncate=False)

KeyboardInterrupt: 

In [4]:
df = spark.read.format("kafka").option("kafka.bootstrap.servers", "kafka-1:9092").option("subscribe", "user_review").load().selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value")

In [5]:
df.show(5, truncate=False)

+-----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key  |value                                                                                                                                                                                                                                                                                                                                                               

In [6]:
user_review_schema = StructType([
    StructField("user_name", StringType(), True),
    StructField("joined_duration", StringType(), True),
    StructField("wrote", StringType(), True),
    StructField("received", FloatType(), True),
    StructField("star_score", StringType(), True),
    StructField("status", TimestampType(), True),
    StructField("comment", IntegerType(), True),
    StructField("images_urls", ArrayType(StringType()), True),
    StructField("rate_time", StringType(), True),
    StructField("use_time", StringType(), True),
    StructField("prod_properties", ArrayType(StringType()), True),
])

In [7]:
df = df.select("key", F.from_json("value", user_review_schema).alias("value")).select("key", "value.*")
df.show(5, truncate=False)

+---+---------+---------------+-----+--------+----------+------+-------+-----------+---------+--------+---------------+
|key|user_name|joined_duration|wrote|received|star_score|status|comment|images_urls|rate_time|use_time|prod_properties|
+---+---------+---------------+-----+--------+----------+------+-------+-----------+---------+--------+---------------+
+---+---------+---------------+-----+--------+----------+------+-------+-----------+---------+--------+---------------+



In [8]:
product_property_df = df.select("key", F.explode("prod_properties").alias("product_property"))

In [9]:
product_property_df.select(F.lower(F.col("product_property"))).show(5, truncate=False)

+-----------------------+
|lower(product_property)|
+-----------------------+
+-----------------------+



In [10]:
from datetime import datetime
from time import perf_counter_ns

start_time = perf_counter_ns()
print(datetime.now())
print("elapsed time: ", (perf_counter_ns() - start_time)/100000000, "seconds")

2026-11-03 21:21:50.586915
elapsed time:  0.002695 seconds


In [11]:
from functools import partial

def greet(greeting, name):
    print(f"{greeting}, {name}!")

hello = partial(greet, "Hello", "nam")
# hello("An")   # in ra: Hello, An!
# hello("Binh") # in ra: Hello, Binh!
hello()

Hello, nam!


## Test load bronze product snapshot data from kafka topic to clickhouse table

In [12]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType, ArrayType
from dotenv import load_dotenv
import os

load_dotenv(override=True)

CLICKHOUSE_URL = "jdbc:clickhouse://clickhouse-server:8123/ecommercial_warehouse"
CLICKHOUSE_TABLE = "bronze_product_snapshot"
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD")

In [13]:
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

In [14]:
product_snapshot_df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka-1:9092")
    .option("subscribe", "product_data_changes")
    .load()
)

In [15]:
product_snapshot_df.show(5, truncate=False)

+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
records_df = product_snapshot_df.selectExpr("CAST(value AS STRING) AS message")
records_df.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
product_snapshot_schema = StructType(
    [
        StructField("urlId", StringType(), True),
        StructField("title", StringType(), True),
        StructField("brand", StringType(), True),
        StructField("price", StringType(), True),
        StructField("discount", StringType(), True),
        StructField("sellPrice", StringType(), True),
        StructField("sold", StringType(), True),
        StructField("productRate", StringType(), True),
        StructField("productRateCount", StringType(), True),
        StructField("categoryTree", StringType(), True),
        StructField("brandOrigin", StringType(), True),
        StructField("madeIn", StringType(), True),
        StructField("warranty", StringType(), True),
        StructField("description", StringType(), True),
        StructField("seller", StringType(), True),
        StructField("sellerRate", StringType(), True),
        StructField("sellerRateCount", StringType(), True),
        StructField("additionalInfo", StringType(), True),
        StructField("observed_at", StringType(), True),
    ]
)

In [18]:
records = records_df.select(F.from_json("message", product_snapshot_schema).alias("data")).select("data.*")

In [19]:
saved_records = records.select(
    F.lower(records.urlId).alias("url_id"),
    F.lower(records.title).alias("title"),
    F.lower(records.brand).alias("brand"),
    F.lower(records.price).alias("price"),
    F.lower(records.discount).alias("discount"),
    F.lower(records.sellPrice).alias("sell_price"),
    F.lower(records.sold).alias("sold"),
    F.lower(records.productRate).alias("product_rate"),
    F.lower(records.productRateCount).alias("product_rate_cnt"),
    F.lower(records.categoryTree).alias("category_tree"),
    F.lower(records.brandOrigin).alias("brand_origin"),
    F.lower(records.madeIn).alias("made_in"),
    F.lower(records.warranty).alias("warranty"),
    F.lower(records.description).alias("description"),
    F.lower(records.seller).alias("seller"),
    F.lower(records.sellerRate).alias("seller_rate"),
    F.lower(records.sellerRateCount).alias("seller_rate_cnt"),
    F.lower(records.additionalInfo).alias("additional_info"),
    F.lower(records.observed_at).alias("observed_at"),
)

In [20]:
saved_records.show(5, truncate=False)

+------+-----------------------------------------------------------------------+-----------+-----+--------+----------+----+------------+----------------+------------------------------------------------------------------------------------------------------+------------+--------+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [21]:
saved_records.write.format("jdbc").option("url", CLICKHOUSE_URL) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("dbtable", CLICKHOUSE_TABLE) \
    .option("user", CLICKHOUSE_USER) \
    .option("password", CLICKHOUSE_PASSWORD) \
    .mode("append") \
    .save()